# 가장 높은 총 수익 Lab
가장 높은 총 수익을 창출하는 3개의 트래픽 소스를 확인하세요.
1. 트래픽 소스별 매출 집계
2. 총 매출 기준 상위 3개 트래픽 소스 가져오기
3. 매출 열을 소수점 둘째 자리까지 정리

##### 메서드
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html" target="_blank">DataFrame</a>: **`groupBy`**, **`sort`**, **`limit`**
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html" target="_blank">Column</a>: **`alias`**, **`desc`**, **`cast`**, **`operators`**
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html" target="_blank">내장 함수</a>: **`avg`**, **`sum`**

In [0]:
%run ../Includes/Classroom-Setup-00.06L

Resetting the learning environment:
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks"...(0 seconds)

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02"

Validating the locally installed datasets:
| listing local files...(4 seconds)
| validation completed...(4 seconds total)

Creating & using the schema "3dt005_nuk5_da_dewd" in the catalog "hive_metastore"...(0 seconds)

Cloning the sales table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/sales_hist...(4 seconds / 10,510 records)
Cloning the events table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/events_hist...(4 seconds / 485,696 records)
Cloning the users table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/users_hist...(4 seconds / 251,501 records)
Cloning the products table from dbfs:

### 설정
아래 셀을 실행하여 시작 DataFrame **`df`**를 만듭니다.

In [0]:
from pyspark.sql.functions import col

# Purchase events logged on the BedBricks website
df = (spark.table("events")
      .withColumn("revenue", col("ecommerce.purchase_revenue_in_usd"))
      .filter(col("revenue").isNotNull())
      .drop("event_name")
     )

display(df.limit(10))

device,ecommerce,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id,revenue
Windows,"List(1795.0, 1, 1)",1593622944339060,1593623248104571,"List(Corpus Christi, TX)","List(List(null, M_PREM_Q, Premium Queen Mattress, 1795.0, 1795.0, 1))",facebook,1593616330759799,UA000000106537762,1795.0
macOS,"List(1045.0, 1, 1)",1593614860452264,1593615004570770,"List(Kearney, NE)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593610742063884,UA000000106509088,1045.0
Windows,"List(535.5, 1, 1)",1593816370279527,1593816433585044,"List(Phoenix, AZ)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1593608758511600,UA000000106500611,535.5
Android,"List(1095.0, 1, 1)",1593620776166531,1593621612951939,"List(Albuquerque, NM)","List(List(null, M_PREM_T, Premium Twin Mattress, 1095.0, 1095.0, 1))",google,1593616245128899,UA000000106537277,1095.0
Windows,"List(940.5, 1, 1)",1593806700141558,1593807430173589,"List(Dallas, TX)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593606856891824,UA000000106493435,940.5
macOS,"List(1995.0, 1, 1)",1593588128101005,1593588169734017,"List(Clearwater, FL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",email,1593585605089308,UA000000106460128,1995.0
macOS,"List(940.5, 1, 1)",1593685522565145,1593686110618872,"List(Kansas City, MO)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593603980046403,UA000000106484209,940.5
macOS,"List(1045.0, 1, 1)",1593602825823702,1593607138856941,"List(Breckenridge, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593598423581032,UA000000106471468,1045.0
iOS,"List(1695.0, 1, 1)",1593616170661327,1593616175423652,"List(Newark, NJ)","List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))",instagram,1593613743747321,UA000000106523740,1695.0
iOS,"List(1045.0, 1, 1)",1593612481929710,1593612973155915,"List(Raleigh, NC)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593610433648747,UA000000106507739,1045.0



### 1. 트래픽 소스별 총 수익
- **`traffic_source`**로 그룹화
- **`revenue`**의 합계를 **`total_rev`**로 구합니다. 소수점 첫째 자리까지 반올림합니다(예: `nnnnn.n`).
- **`revenue`**의 평균을 **`avg_rev`**로 구합니다.

필요한 내장 함수를 모두 import하는 것을 잊지 마세요.

In [0]:
# TODO
from pyspark.sql.functions import sum, avg, round

traffic_df = (df
    .groupBy("traffic_source")
    .agg(round(sum("revenue"), 1).alias("total_rev"),
         avg("revenue").alias("avg_rev"))
)

display(traffic_df.limit(10))

traffic_source,total_rev,avg_rev
instagram,826921.0,1086.6241787122208
direct,620096.0,1049.2318104906938
youtube,404911.0,1091.4043126684637
email,4026578.5,986.1813617438238
facebook,1200591.0,1067.192
google,2322856.0,1093.108705882353



**1.1: CHECK YOUR WORK**

In [0]:
from pyspark.sql.functions import round

expected1 = [(620096.0, 1049.2318), (4026578.5, 986.1814), (1200591.0, 1067.192), (2322856.0, 1093.1087), (826921.0, 1086.6242), (404911.0, 1091.4043)]
test_df = traffic_df.sort("traffic_source").select(round("total_rev", 4).alias("total_rev"), round("avg_rev", 4).alias("avg_rev"))
result1 = [(row.total_rev, row.avg_rev) for row in test_df.collect()]

assert(expected1 == result1)
print("All test pass")

All test pass



### 2. 총 수익 기준 상위 3개 트래픽 소스 가져오기
- **`total_rev`**를 기준으로 내림차순 정렬
- 처음 3개 행으로 제한

In [0]:
# TODO
top_traffic_df = (traffic_df
    .sort(col("total_rev").desc())
    .limit(3)
)
display(top_traffic_df)

traffic_source,total_rev,avg_rev
email,4026578.5,986.1813617438238
google,2322856.0,1093.108705882353
facebook,1200591.0,1067.192



**2.1: CHECK YOUR WORK**

In [0]:
expected2 = [(4026578.5, 986.1814), (2322856.0, 1093.1087), (1200591.0, 1067.192)]
test_df = top_traffic_df.select(round("total_rev", 4).alias("total_rev"), round("avg_rev", 4).alias("avg_rev"))
result2 = [(row.total_rev, row.avg_rev) for row in test_df.collect()]

assert(expected2 == result2)
print("All test pass")

All test pass



### 3. 매출 열의 소수점 이하 두 자리까지 제한
- **`avg_rev`** 및 **`total_rev`** 열을 수정하여 소수점 이하 두 자리까지 포함하는 숫자를 포함하도록 합니다.
- 동일한 이름의 **`withColumn()`**을 사용하여 이러한 열을 바꿉니다.
- 소수점 이하 두 자리까지 제한하려면 각 열에 100을 곱하고 long으로 변환한 후 100으로 나눕니다.

In [0]:
# TODO
final_df = (top_traffic_df
    .withColumn("avg_rev", (col("avg_rev") * 100).cast("long") / 100)
    .withColumn("total_rev", (col("total_rev") * 100).cast("long") / 100)
)

display(final_df.limit(10))

traffic_source,total_rev,avg_rev
email,4026578.5,986.18
google,2322856.0,1093.1
facebook,1200591.0,1067.19



**3.1: CHECK YOUR WORK**

In [0]:
expected3 = [(4026578.5, 986.18), (2322856.0, 1093.1), (1200591.0, 1067.19)]
result3 = [(row.total_rev, row.avg_rev) for row in final_df.collect()]

assert(expected3 == result3)
print("All test pass")

All test pass


### 4. 보너스: 내장 수학 함수를 사용하여 다시 작성하세요.
지정된 소수점 이하 자릿수로 반올림하는 내장 수학 함수를 찾으세요.

In [0]:
# TODO
bonus_df = (top_traffic_df
    .withColumn("avg_rev", round(col("avg_rev"), 2))
    .withColumn("total_rev", round(col("total_rev"), 2))
)

display(bonus_df)

traffic_source,total_rev,avg_rev
email,4026578.5,986.18
google,2322856.0,1093.11
facebook,1200591.0,1067.19



**4.1: CHECK YOUR WORK**

In [0]:
expected4 = [(4026578.5, 986.18), (2322856.0, 1093.11), (1200591.0, 1067.19)]
result4 = [(row.total_rev, row.avg_rev) for row in bonus_df.collect()]

assert(expected4 == result4)
print("All test pass")

All test pass



### 5. 위의 모든 단계를 연결하세요

In [0]:
# TODO
from pyspark.sql.functions import sum, avg, round

chain_df = (df
    .groupBy("traffic_source")
    .agg(round(sum("revenue"), 1).alias("total_rev"),
         avg("revenue").alias("avg_rev"))
    .sort(col("total_rev").desc())
    .limit(3)
    .withColumn("avg_rev", round(col("avg_rev"), 2))
    .withColumn("total_rev", round(col("total_rev"), 2))
)

display(chain_df)

traffic_source,total_rev,avg_rev
email,4026578.5,986.18
google,2322856.0,1093.11
facebook,1200591.0,1067.19



**5.1: CHECK YOUR WORK**

In [0]:
expected5 = [(4026578.5, 986.18), (2322856.0, 1093.11), (1200591.0, 1067.19)]
result5 = [(row.total_rev, row.avg_rev) for row in chain_df.collect()]

assert(expected5 == result5)
print("All test pass")

All test pass



이 레슨과 관련된 테이블과 파일을 삭제하려면 다음 셀을 실행하세요.

In [0]:
DA.cleanup()

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_dewd"...(2 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks"...(1 seconds)

Validating the locally installed datasets:
| listing local files...(4 seconds)
| validation completed...(4 seconds total)
